### Tools
Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:

A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
A function or coroutine to execute.

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")  
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.tools import tool

model=ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")

@tool
def getWeather(city:str)->str:
    """
    Get the current weather for a given city.
    """
    return f"The current weather in {city} is sunny with a temperature of 25°C."

model_with_tools=model.bind_tools([getWeather])


In [9]:
response=model_with_tools.invoke("What is the weather in New York City?")
print(response)
for tool_call in response.tool_calls:
    print(f"Tool called: {tool_call['name']}")
    print(f"Tool argument: {tool_call['args']}")

content=[] additional_kwargs={'function_call': {'name': 'getWeather', 'arguments': '{"city": "New York City"}'}, '__gemini_function_call_thought_signatures__': {'call_1745179': 'EmAKXgFpFH0TBGEtIqnGghfD4pr471vhkuwT0eP16IvyG7X0ElCzFqDpcnhkPW5Il9LLrezUB3CagMjIRYNSXGPlmAySeWCUHMhXvmqDpNQFQjEoj2U2V9am1s7KolpOmNc='}} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--01a11719-d0a6-7483-a0a9-e556225770c0-0' tool_calls=[{'name': 'getWeather', 'args': {'city': 'New York City'}, 'id': 'call_1745179', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 55, 'output_tokens': 17, 'total_tokens': 72, 'input_token_details': {'cache_read': 0}}
Tool called: getWeather
Tool argument: {'city': 'New York City'}


## Tool execution loops

In [7]:
## tool execution loops
msg=[{"role":"user","content":"what is the weather in New York City?"}]
ai_msg=model_with_tools.invoke(msg)
msg.append(ai_msg)


for tool_call in ai_msg.tool_calls:
    tool_result=getWeather.invoke(tool_call)
    msg.append(tool_result)

final_result=model_with_tools.invoke(msg)
print(final_result.text)
print(msg)

The current weather in New York City is sunny with a temperature of 25°C.
[{'role': 'user', 'content': 'what is the weather in New York City?'}, AIMessage(content=[], additional_kwargs={'function_call': {'name': 'getWeather', 'arguments': '{"city": "New York City"}'}, '__gemini_function_call_thought_signatures__': {'call_440019': 'EmAKXgFpFH0TGxkV+cGjQsSIxpmlLcsx7xqDR2MPWYrsk/jtKHRqifhoj8sLQhB2tv665TVpcHhBbZkRy78jhmUWBLoyrqE8nyZJdLLGcipjVAw17I5Qq32Dx5LWLY8ENXw='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a11a2b-a811-7211-bd2b-146c4f1e80c1-0', tool_calls=[{'name': 'getWeather', 'args': {'city': 'New York City'}, 'id': 'call_440019', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 55, 'output_tokens': 17, 'total_tokens': 72, 'input_token_details': {'cache_read': 0}}), ToolMessage(content='The current weather in New York City is sunny with a tempe